In [1]:
!pip install scikit-learn xgboost

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

In [3]:
BASE = "/kaggle/input/notebooks/navishaaagarwaal/02-feature-extraction"
BASE2 = "/kaggle/input/notebooks/navishaaagarwaal/01-dataset-preparation"
text_emb = np.load(BASE + "/text_emb.npy")
img_emb = np.load(BASE + "/img_emb.npy")
hash_emb = np.load(BASE + "/hash_emb.npy")

hashtag_count = np.load(BASE + "/hashtag_count.npy")
hashtag_freq = np.load(BASE + "/hashtag_freq.npy")

df = pd.read_csv(BASE2 + "/labeled.csv")

y = df["label"].values

In [4]:
X_train_idx, X_test_idx = train_test_split(
    np.arange(len(y)),
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [5]:
X = text_emb

X_train = X[X_train_idx]
X_test = X[X_test_idx]

y_train = y[X_train_idx]
y_test = y[X_test_idx]

clf = LogisticRegression(max_iter=2000, class_weight="balanced")

clf.fit(X_train, y_train)

pred = clf.predict(X_test)
prob = clf.predict_proba(X_test)[:,1]

print("TEXT MODEL")
print(classification_report(y_test, pred))
print("ROC-AUC:", roc_auc_score(y_test, prob))

TEXT MODEL
              precision    recall  f1-score   support

           0       0.96      0.81      0.88      2083
           1       0.36      0.75      0.49       298

    accuracy                           0.80      2381
   macro avg       0.66      0.78      0.68      2381
weighted avg       0.88      0.80      0.83      2381

ROC-AUC: 0.843580341982234


In [6]:
X = img_emb

X_train = X[X_train_idx]
X_test = X[X_test_idx]

clf = LogisticRegression(max_iter=2000, class_weight="balanced")

clf.fit(X_train, y_train)

pred = clf.predict(X_test)
prob = clf.predict_proba(X_test)[:,1]

print("IMAGE MODEL")
print(classification_report(y_test, pred))
print("ROC-AUC:", roc_auc_score(y_test, prob))

IMAGE MODEL
              precision    recall  f1-score   support

           0       0.87      1.00      0.93      2083
           1       0.00      0.00      0.00       298

    accuracy                           0.87      2381
   macro avg       0.44      0.50      0.47      2381
weighted avg       0.77      0.87      0.82      2381

ROC-AUC: 0.5


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [7]:
X = np.concatenate([text_emb, img_emb], axis=1)

X_train = X[X_train_idx]
X_test = X[X_test_idx]

neg = np.sum(y_train == 0)
pos = np.sum(y_train == 1)

clf = XGBClassifier(
    eval_metric="logloss",
    tree_method="hist",
    scale_pos_weight=neg/pos,
    random_state=42
)

clf.fit(X_train, y_train)

pred = clf.predict(X_test)
prob = clf.predict_proba(X_test)[:,1]

print("TEXT + IMAGE MODEL")
print(classification_report(y_test, pred))
print("ROC-AUC:", roc_auc_score(y_test, prob))

TEXT + IMAGE MODEL
              precision    recall  f1-score   support

           0       0.92      0.96      0.94      2083
           1       0.58      0.40      0.47       298

    accuracy                           0.89      2381
   macro avg       0.75      0.68      0.70      2381
weighted avg       0.88      0.89      0.88      2381

ROC-AUC: 0.8241283706064111


In [8]:
count = hashtag_count.reshape(-1,1)
freq = hashtag_freq.reshape(-1,1)

X = np.concatenate([
    text_emb,
    img_emb,
    hash_emb,
    count,
    freq
], axis=1)

X_train = X[X_train_idx]
X_test = X[X_test_idx]

clf = XGBClassifier(
    eval_metric='logloss',
    tree_method='hist',
    random_state=42
)

clf.fit(X_train, y_train)

pred = clf.predict(X_test)
prob = clf.predict_proba(X_test)[:,1]

print("FULL MULTIMODAL MODEL")
print(classification_report(y_test, pred))
print("ROC-AUC:", roc_auc_score(y_test, prob))

FULL MULTIMODAL MODEL
              precision    recall  f1-score   support

           0       0.90      0.98      0.94      2083
           1       0.67      0.27      0.38       298

    accuracy                           0.89      2381
   macro avg       0.79      0.62      0.66      2381
weighted avg       0.87      0.89      0.87      2381

ROC-AUC: 0.8230683352289385
